# Backtest + MLflow: compare combinations

This notebook compares a parameter grid on earlier dates, selects one combination, then evaluates **only the winner** on held-out recent dates. MLflow records the evidence and provides a local comparison UI. Start with the synthetic demo: no real input or EEX files are needed. Parameters are not automatically applied to your configuration or production curves.

[Spanish guide](../MLFLOW.es.md) / [Full English guide](../MLFLOW.md)

**Install**, from the repository root:
```powershell
uv sync --group notebook --group experiment
uv run --group notebook --group experiment jupyter lab notebooks/backtest_mlflow.ipynb
```
Select the local `.venv` kernel in Jupyter or VS Code, then run cells in order. Code and comments are in English.


## 1. Data, dates and service

Edit this cell. `None` discovers the root/config and uses the available date bounds. Relative paths in this cell are relative to the project root; paths **inside the TOML** retain their usual TOML-relative rules. `VALIDATION_DAYS=5` reserves five eligible observation dates, not necessarily five calendar days. In real mode, `VWAP_PATH=None` uses the configured input. Example paths/dates are not requirements.

`EEX_OFFSET_DAYS=None` keeps `[eex].offset_days` from the TOML (default `0`). Set `-1` to allow only publications on or before the previous calendar day, or `-2` for two calendar days earlier. The most recent available publication within that cutoff is used; weekends can make its actual age larger. Staleness is measured from the original reference date. This is a fixed evaluation scenario, not a parameter-grid axis: compare different offsets in separate runs.


In [ ]:
PROJECT_ROOT = None  # Example: r"C:\Projects\VWAPS".
CONFIG_PATH = None  # Default: PROJECT_ROOT / "config.toml".
DATA_MODE = "synthetic"  # "synthetic" or "real".
VWAP_PATH = None  # Real mode only; example: r"C:\Data\VWAPS\observations.xlsx".
START_DATE = None  # Inclusive ISO date, e.g. "2026-08-10".
END_DATE = None  # Inclusive ISO date, e.g. "2026-10-06".
EEX_OFFSET_DAYS = None  # Keep the TOML setting; use 0 for same-day availability or -1 for the previous calendar day.
VALIDATION_DAYS = 5
MAX_TRIALS = 24
DEMO_SEED = 7
DEMO_PERIODS = 30
MLFLOW_PORT = 5000
MLFLOW_STORAGE = "output/mlflow"
EXPERIMENT_NAME = "vwaps-refill"
LOG_PREDICTIONS = True

## 2. What to compare

The initial grid has **3 modes x 2 history settings = 6 combinations**. Edit the lists or choose a `GRID_EXAMPLES` entry. The notebook supports model controls, historical memory, fallback and shape: see the exact names and limits in the [parameter table](../MLFLOW.md). Omitted fields stay fixed. Paths, mapping, targets, conventions, observation filters and ratio guards are excluded to preserve the comparison universe. The CLI `tune` command retains its original six controls.

Test one question at a time. The examples are research candidates, not established recommendations. If LOCAL/HIST fills every hidden case, fallback settings can tie because fallback is unused. The `fallback_isolated` example explicitly disables local, hist and cross to study that branch; it does not represent the full production configuration. `shape_mode="audit"` and the diagnostic tolerance do not change prices.


In [ ]:
from copy import deepcopy

GRID_EXAMPLES = {
    "basis_only": {"basis_mode": ["auto", "ratio", "additive"]},
    "basis_and_history": {
        "basis_mode": ["auto", "ratio", "additive"],
        "layer_hist": ["off", "auto"],
    },
    "distance_and_shrinkage": {
        "tau_log": [0.35, 0.75, 1.5],
        "shrink_k": [0.5, 2.0, 8.0],
    },
    "local_reach": {
        "tau_log": [0.25, 0.5, 1.0],
        "other_kind_weight": [0.2, 0.6],
    },
    "historical_memory": {
        "layer_hist": ["on"],
        "ewma_halflife_days": [5.0, 10.0, 20.0],
        "hist_max_age_days": [30, 60],
    },
    "fallback_averaging": {
        "fallback_price_method": ["simple", "ewma"],
        "fallback_price_window": [3, 5, 9],
    },
    "fallback_isolated": {
        "layer_local": [False],
        "layer_hist": ["off"],
        "layer_cross": [False],
        "fallback_price_method": ["simple", "ewma"],
        "fallback_price_window": [3, 5, 9],
    },
    "shape_coherence": {
        "shape_mode": ["off", "adjust"],
        "shape_adjust_originals": [False],
        "shape_coherence_weight": [1.0, 10.0],
    },
    "optional_learned_layers": {
        "layer_correlation": [False, True],
        "layer_cross": [False, True],
    },
}

PARAMETER_GRID = {
    "basis_mode": ["auto", "ratio", "additive"],
    "layer_hist": ["off", "auto"],
}
# Alternative: PARAMETER_GRID = deepcopy(GRID_EXAMPLES["distance_and_shrinkage"])

## 3. Prepare environment and data

The demo generates observations, mapping and EEX in memory using an isolated configuration copy. Real mode loads your files. An explicit `EEX_OFFSET_DAYS` overrides the in-memory configuration after either dataset is prepared; the TOML remains unchanged. Tuning requires `run.warmup_days=0`; it is not automatically rewritten. The evaluation range does not discard supplied earlier history. Check the displayed dates, EEX availability policy and combination count before starting the experiment.


In [ ]:
import json
import math
import sys
from dataclasses import replace
from datetime import date, datetime, timezone
from pathlib import Path
from uuid import uuid4

import pandas as pd
from IPython.display import Markdown, display

if PROJECT_ROOT is None:
    search_root = Path.cwd().resolve()
    candidates = [search_root, *search_root.parents]
    ROOT = next(
        (path for path in candidates
         if (path / "pyproject.toml").is_file() and (path / "src/vwaps").is_dir()),
        None,
    )
    if ROOT is None:
        raise ValueError("Set PROJECT_ROOT to the repository directory.")
else:
    ROOT = Path(PROJECT_ROOT).expanduser().resolve()
    if not (ROOT / "src/vwaps").is_dir():
        raise ValueError("PROJECT_ROOT must contain src/vwaps.")


def project_path(value):
    """Resolve a notebook path relative to the repository root."""
    path = Path(value).expanduser()
    return path.resolve() if path.is_absolute() else (ROOT / path).resolve()


source_path = str(ROOT / "src")
if source_path not in sys.path:
    sys.path.insert(0, source_path)

from vwaps.config import load_config, validate_config
from vwaps.experiment_data import load_experiment_dataset, make_demo_dataset
from vwaps.experiments import run_tracked_tuning
from vwaps.mlflow_server import start_mlflow_server
from vwaps.tuning import EXPERIMENT_TUNABLE_FIELDS, TUNABLE_FIELDS

unknown_fields = set(PARAMETER_GRID) - set(EXPERIMENT_TUNABLE_FIELDS)
if unknown_fields:
    raise ValueError(f"Unsupported grid fields: {sorted(unknown_fields)}")
if any(not isinstance(values, list) or not values for values in PARAMETER_GRID.values()):
    raise ValueError("Every grid field needs a nonempty list of candidate values.")
if isinstance(MAX_TRIALS, bool) or not isinstance(MAX_TRIALS, int) or MAX_TRIALS < 1:
    raise ValueError("MAX_TRIALS must be a positive integer.")
trial_count = math.prod(len(values) for values in PARAMETER_GRID.values())
if trial_count > MAX_TRIALS:
    raise ValueError(f"Grid has {trial_count} combinations; MAX_TRIALS={MAX_TRIALS}.")
if DATA_MODE not in ("synthetic", "real"):
    raise ValueError("DATA_MODE must be 'synthetic' or 'real'.")

config_path = project_path(CONFIG_PATH or "config.toml")
storage_dir = project_path(MLFLOW_STORAGE)
base_config = load_config(config_path)
if DATA_MODE == "synthetic":
    dataset = make_demo_dataset(base_config, seed=DEMO_SEED, periods=DEMO_PERIODS)
else:
    input_override = None if VWAP_PATH is None else project_path(VWAP_PATH)
    dataset = load_experiment_dataset(base_config, vwap_path=input_override)
if EEX_OFFSET_DAYS is not None:
    dataset = replace(dataset, config=replace(dataset.config, eex_offset_days=EEX_OFFSET_DAYS))
validate_config(dataset.config)
if dataset.config.warmup_days != 0:
    raise ValueError("Tuning requires run.warmup_days=0 in the selected configuration.")
start = dataset.start if START_DATE is None else date.fromisoformat(START_DATE)
end = dataset.end if END_DATE is None else date.fromisoformat(END_DATE)
if start > end:
    raise ValueError("START_DATE must not be later than END_DATE.")

print(f"Python: {sys.executable}")
print(f"Project: {ROOT}")
print(f"Config: {config_path}")
print(f"Dataset: {dataset.label}; observation rows: {len(dataset.vwaps):,}")
print(f"Evaluation: {start} through {end}; latest {VALIDATION_DAYS} eligible dates held out")
print(f"EEX availability: offset {dataset.config.eex_offset_days} calendar days; latest publication <= reference date + offset")
print(f"EEX staleness: measured from the reference date; maximum age: {dataset.config.max_stale_days} (None means unlimited)")
print(f"Grid: {trial_count} combinations; limit: {MAX_TRIALS}")
print(f"Supported experiment fields: {list(EXPERIMENT_TUNABLE_FIELDS)}")
print(f"Storage: {storage_dir}")
display(pd.DataFrame([
    {"parameter": name, "candidate_values": PARAMETER_GRID.get(name, [getattr(dataset.config, name)]),
     "varied_in_grid": name in PARAMETER_GRID}
    for name in dict.fromkeys([*TUNABLE_FIELDS, *PARAMETER_GRID])
]))

## 4. What winning means

We hide one own observation and aliases of its physical delivery period; other own prices on that date remain available. The truth is **your hidden price**, not EEX. Maximum coverage comes first. Among candidates with that coverage, the lowest equal-curve mean `MAE_model / MAE_EEX` wins, measured on predictions shared by **all** calibration candidates. Absolute errors are separated by unit. Only the winner is validated; its parameters stay fixed while earlier validation observations can feed later history.

This does not simulate every real missing block. The demo demonstrates the workflow, not your best real parameters. If you repeatedly revise the grid after looking at validation, reserve other untouched dates for a final check.


In [ ]:
server = start_mlflow_server(storage_dir, port=MLFLOW_PORT, timeout=45)
display(Markdown(f"**Local MLflow:** [{server.url}]({server.url})"))
print(f"Service log: {server.log_path}")

## 5. Run and track

This cell creates a new run. Reports go into a unique directory under `output/mlflow` (or your selected storage path). It does not change configuration or production curve CSVs. With `LOG_PREDICTIONS=True`, experiment artifacts may include real own observations, stored in the selected local directory. The service listens on `127.0.0.1`; no cloud account is needed.

`LOG_PREDICTIONS=False` omits individual paired price/prediction artifacts; aggregate reports and metadata are still recorded.


In [ ]:
report_name = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid4().hex[:8]
report_dir = storage_dir / "reports" / report_name
result = run_tracked_tuning(
    dataset.config,
    dataset.vwaps,
    dataset.maps,
    dataset.books,
    start,
    end,
    PARAMETER_GRID,
    VALIDATION_DAYS,
    MAX_TRIALS,
    tracking_uri=server.url,
    experiment_name=EXPERIMENT_NAME,
    output_dir=report_dir,
    data_label=dataset.label,
    log_predictions=LOG_PREDICTIONS,
)
display(Markdown(f"**MLflow run:** [Open run]({result.run_url})"))
print(f"Run ID: {result.run_id}")
print(f"Experiment ID: {result.experiment_id}")
print(f"Reports: {result.output_dir}")

## 6. Calibration ranking

`eligible_for_selection=True` marks maximum coverage; `selected=True` identifies the winner. Lower `score` is better: <1 beats EEX under the equal-curve measure; >1 is worse. It is not a percentage price error. If EEX is exact, the ratio is 1 when the model is also exact and infinity otherwise. The table puts eligible candidates first, then score and grid order for ties. Inspect `coverage` and `n_paired` alongside score.


In [ ]:
calibration = result.tuning.calibration_report.copy()
ranking = calibration.loc[calibration["scope"].eq("overall")].sort_values(
    ["eligible_for_selection", "score", "trial_id"],
    ascending=[False, True, True],
    kind="stable",
)
ranking_columns = [
    "trial_id", "selected", "eligible_for_selection", "coverage", "n_baseline",
    "n_available", "n_missing", "n_paired", "n_curves", "n_paired_dates",
    "score", "normalized_skill", *dict.fromkeys([*TUNABLE_FIELDS, *PARAMETER_GRID]),
]
display(ranking[ranking_columns].reset_index(drop=True))
display(Markdown("**Calibration errors by unit**"))
display(calibration.loc[calibration["scope"].eq("unit")].reset_index(drop=True))

## 7. Proposal and winner validation

These parameters are a proposal, not automatically applied. The table below contains **only the selected combination** on the reserved dates. Overall absolute MAE/RMSE/bias fields are blank to avoid mixing units; inspect the unit rows. A calibration winner may fail on validation. A successful demo does not validate filling your real missing observations.


In [ ]:
print("Proposed parameters (not applied):")
print(json.dumps(result.tuning.selected_config, indent=2))
display(result.tuning.validation_report.reset_index(drop=True))
display(pd.DataFrame([
    {"item": key, "value": str(value)}
    for key, value in result.tuning.metadata.items()
]))

## 8. Inspect a saved model's full curves

Choose an experiment, an earlier run, a trial and a saved stage, then click **Load saved curves**. The shared viewer offers product/region/unit selection, all contract types together or one family, a single reference date, paired means over a date range and fixed-delivery evolution. It compares final prices with the EEX reference saved in that experiment, including optional own-price and pre-shape traces.

**These are full refill curves with own observations visible.** A curve passing through an original price does not prove prediction accuracy. Accuracy is measured by hiding an own observation and comparing its predicted value with that hidden own value. Selection first maximizes calibration coverage; among those candidates, the lowest equal-curve mean of `MAE_model / MAE_EEX` wins on common cases. A smoother shape is **not** the selection objective. Only the winning trial receives validation curves; other trials have calibration curves only.

To browse previous experiments without rerunning data preparation or tuning, run **the editable settings cell** and **the viewer cell below**. That cell starts or reuses the local MLflow service and reads saved artifacts. You do not need the original input or EEX files. `LOG_PREDICTIONS=False`, older runs or interrupted trials may have no saved curves; the viewer explains this and never rebuilds old results using current data. Temporary downloads are placed under `MLFLOW_STORAGE / "viewer_cache"` and removed after loading.


In [ ]:
from pathlib import Path
import sys
from IPython.display import Markdown, display

# This cell can run after editable settings, without preparing data or rerunning tuning.
viewer_search = Path(PROJECT_ROOT).expanduser().resolve() if PROJECT_ROOT else Path.cwd().resolve()
viewer_root = next(
    (path for path in (viewer_search, *viewer_search.parents)
     if (path / "pyproject.toml").is_file() and (path / "src/vwaps").is_dir()),
    None,
)
if viewer_root is None:
    raise ValueError("Set PROJECT_ROOT to the repository directory.")
if str(viewer_root / "src") not in sys.path:
    sys.path.insert(0, str(viewer_root / "src"))

from vwaps.experiment_browser import create_experiment_browser
from vwaps.mlflow_server import start_mlflow_server

viewer_storage = Path(MLFLOW_STORAGE).expanduser()
viewer_storage = viewer_storage if viewer_storage.is_absolute() else viewer_root / viewer_storage
server = start_mlflow_server(viewer_storage.resolve(), port=MLFLOW_PORT, timeout=45)
EXPERIMENT_BROWSER = create_experiment_browser(
    server.url,
    experiment_name=EXPERIMENT_NAME,
    selected_parent_run_id=getattr(globals().get("result"), "run_id", None),
    cache_dir=viewer_storage / "viewer_cache",
)
display(EXPERIMENT_BROWSER)


## 9. Keep or stop the UI

Leave `STOP_SERVER=False` to use MLflow after Run All. Set it to `True` and execute this cell to stop the service; stored experiments are kept. Reexecuting startup in the **same kernel** reuses its owned service. After a kernel restart, the new process does not adopt an earlier service: normal cleanup stops the owned service and records remain on disk. If an earlier or unrelated service still occupies the port, choose another port or stop it from its owning session. Inspect `server.log_path` if startup fails.


In [ ]:
STOP_SERVER = False
if STOP_SERVER:
    server.stop()
    print("Managed MLflow service stopped; stored runs were kept.")
else:
    display(Markdown(f"**MLflow remains available:** [{server.url}]({server.url})"))